# TrainLM T1: TorchTPU single-device probe (v5e-1)

Run this notebook in Kaggle after selecting a **TPU v5e-1** runtime with the owner-approved TorchTPU image. It checks package provenance, forward execution, autograd, an AdamW update, and optionally `torch.compile`.

This notebook does not install packages or guess a TorchTPU module/device name. Enter the device string documented by the runtime. Standard Colab is suitable only if it provides the same supported TorchTPU runtime; a generic accelerator selection is not enough.

v5e-1 cannot validate collectives, DP8, sharding/FSDP, distributed checkpoints, or scaling. Use v5e-8 for checks that require multiple devices.

In [ ]:
from pathlib import Path
import subprocess

PROBE_RELATIVE_PATH = Path("examples/torchtpu_t1_v5e1_probe.py")
repo_candidates = [
    Path.cwd(), *Path.cwd().parents,
    Path("/kaggle/working/TrainLM"), Path("/content/TrainLM"),
]
repo_root = next(
    (candidate.resolve() for candidate in repo_candidates
     if (candidate / PROBE_RELATIVE_PATH).is_file()),
    None,
)

if repo_root is None:
    if Path("/kaggle/working").exists():
        clone_path = Path("/kaggle/working/TrainLM")
    elif Path("/content").exists():
        clone_path = Path("/content/TrainLM")
    else:
        clone_path = Path.cwd() / "TrainLM"
    if clone_path.exists():
        raise FileNotFoundError(
            f"{clone_path} exists but does not contain {PROBE_RELATIVE_PATH}; "
            "attach the TrainLM repository or choose an empty clone path."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/Dhiraj309/TrainLM.git", str(clone_path)],
        check=True,
    )
    repo_root = clone_path.resolve()

assert (repo_root / PROBE_RELATIVE_PATH).is_file()
print(f"repository={repo_root}")

## Choose the runtime's documented device string

Do not guess an alias such as `tpu`. Copy the exact public device string from the installed Kaggle runtime's documentation or setup output. The script refuses CPU unless the separate `--allow-cpu` flag is used for a local-only check.

In [ ]:
DEVICE = input("Runtime-documented PyTorch device string: ").strip()
if not DEVICE:
    raise ValueError("Enter the device string documented by this runtime.")
print(f"selected_device={DEVICE}")

## Eager single-device smoke

The probe uses a tiny synthetic model and data. It prints Python/PyTorch and installed Torch/TPU/XLA-related distribution versions, then checks finite loss and gradients plus an actual optimizer update. Preserve the complete cell output as evidence.

In [ ]:
import sys
import subprocess

probe = repo_root / PROBE_RELATIVE_PATH
subprocess.run(
    [sys.executable, str(probe), "--device", DEVICE],
    cwd=repo_root,
    check=True,
)

## Optional compile smoke

Run this separately because compilation can take longer and may expose a runtime/compiler issue. If it fails, keep the error as evidence; the script does not silently fall back to eager execution.

In [ ]:
RUN_COMPILE = False  # Set True to run the optional torch.compile check.
if RUN_COMPILE:
    subprocess.run(
        [sys.executable, str(probe), "--device", DEVICE, "--compile"],
        cwd=repo_root,
        check=True,
    )
else:
    print("compile probe skipped; set RUN_COMPILE = True when ready")

## Record results

Record the Kaggle image/runtime identifier, date, accelerator, entered device string, command, Python/PyTorch/distribution versions, and whether import, device placement, forward, autograd, optimizer update, and optional compile passed. Do not paste tokens, credentials, or a full environment dump.

This is only a v5e-1 single-device smoke. Move to v5e-8 for collectives, DP8, sharding/FSDP, distributed-checkpoint behavior, or scaling; do not infer those results from this notebook.